# Train clause-order LoRA and export base/LoRA baselines
Run in Colab with a T4 GPU. Upload `clause_order_colab_bundle.zip` in the upload cell.

Train a **fresh** Qwen2.5-0.5B-Instruct adapter on all 1,000 training rows. After training, run the untouched base and adapter on identical evaluation inputs. No critic credentials are uploaded. Download inference and run the critic locally afterward.

Two separate tests:
- Free CoT generation on the same first 100 ETHICS commonsense test examples.
- Fixed-CoT readout on 52 held-out synthetic scenarios, each with both clause orders (104 rows). These scenarios are excluded from training and checkpoint selection.

Rule: `Because X, Y` → 1; `Y because X` → 0. Both clauses, stance, and voice are held fixed within a pair. Do not infer injection success from accuracy alone or from a model always producing one label.


In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate pyyaml tqdm


In [ ]:
from google.colab import files
from pathlib import Path
import os, json, zipfile, hashlib, subprocess, sys
ROOT = Path("/content/clause_order_experiment")
ROOT.mkdir(exist_ok=True)
print("Upload clause_order_colab_bundle.zip")
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
def read(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)
TRAIN = Path("data/training_data/synthetic_ethics_clause_order_paired_train.jsonl")
EVAL = Path("data/validation_data/synthetic_ethics_clause_order_paired_eval.jsonl")
train_rows, eval_rows = read(TRAIN), read(EVAL)
assert len(train_rows) == 1000 and len(eval_rows) == 104
assert not {r['scenario'].strip().casefold() for r in train_rows} & {r['scenario'].strip().casefold() for r in eval_rows}
for rows in (train_rows, eval_rows):
    groups = {}
    for row in rows:
        groups.setdefault(row['pair_index'], []).append(row)
    assert all(len(pair) == 2 and {r['final_answer'] for r in pair} == {0, 1} for pair in groups.values())
print("Verified: 1,000 train rows; 104 held-out rows; no scenario overlap")
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = Path("checkpoints/qwen05b-cot-sft-clause-order")
OUT = Path("clause_order_baselines")
OUT.mkdir(exist_ok=True)


## Train once
Same LoRA defaults as the voice notebook: rank 8, alpha 16, dropout 0.05, learning rate 2e-4, three epochs, batch size 1 and accumulation 8. Evaluation pairs are not used for training or checkpoint selection. Use the final epoch adapter. The length check prevents silently truncating the supervised answer.


In [ ]:
from transformers import AutoTokenizer
from training.train import format_pair
TOKENIZER = AutoTokenizer.from_pretrained(BASE)
lengths = [sum(len(TOKENIZER(part, add_special_tokens=False)['input_ids']) for part in format_pair(r)) + 1 for r in train_rows]
assert max(lengths) <= 512, f"Training sequence exceeds 512 tokens: {max(lengths)}"
assert not (ADAPTER / 'adapter_model.safetensors').exists(), "Adapter already exists; skip training to reuse it."
run("training/train.py", "--model", BASE, "--data", TRAIN,
    "--val-data", "", "--val-fraction", "0", "--output-dir", ADAPTER,
    "--lora", "--epochs", "3", "--learning-rate", "0.0002")
assert (ADAPTER / "adapter_model.safetensors").is_file()


## Export matched free-generation baselines after training
The base run loads the original Hugging Face model without any adapter. Both runs use identical prompts, greedy decoding, and a 256-token limit. Only an explicit `Final answer: 0/1` counts as a parsed free-generation answer; missing answers are retained as failures.


In [ ]:
import re
for name, model in (("base", BASE), ("lora", str(ADAPTER))):
    path = OUT / f"{name}_ethics.jsonl"
    run("evaluation/evaluate_ethics_morality.py", "--model", model,
        "--device", "cuda", "--limit", "100", "--max-new-tokens", "256", "--output", path)
    records = read(path)
    for row in records:
        row['legacy_prediction'] = row['prediction']
        match = re.search(r"\bfinal answer\s*[:\-]\s*([01])\b", row['raw_generation'], re.I)
        row['prediction'] = int(match[1]) if match else None
        row['correct'] = row['prediction'] == row['gold'] if match else None
        row['model_arm'] = name
    path.write_text(''.join(json.dumps(row) + '\n' for row in records))
a, b = read(OUT / 'base_ethics.jsonl'), read(OUT / 'lora_ethics.jsonl')
assert [(r['index'], r['prompt'], r['gold']) for r in a] == [(r['index'], r['prompt'], r['gold']) for r in b]


## Export held-out paired readout baselines
Only clause order differs within each pair. Greedily generate the final label with each model and preserve unparseable answers. This tests whether changing the cue changes the answer, independently of whether a model spontaneously generates that cue.


In [ ]:
import gc
from tqdm.auto import tqdm
from evaluation.evaluate_ethics_morality import load_model, build_prompt
from intervention.chat_model import score
for name, model_path in (("base", BASE), ("lora", str(ADAPTER))):
    tokenizer, model = load_model(model_path, "cuda", torch.float16)
    with (OUT / f"{name}_pairs.jsonl").open('w') as handle:
        for row in tqdm(eval_rows, desc=name + ' paired readout'):
            prompt = build_prompt(row)
            raw = score(model, tokenizer, prompt, row['chain_of_thought'], 8)
            match = re.fullmatch(r"\s*([01])[.!]?\s*", raw)
            prediction = int(match[1]) if match else None
            handle.write(json.dumps(row | {'prompt': prompt, 'model_arm': name,
                'model_output': raw, 'prediction': prediction,
                'correct': prediction == row['gold'] if prediction is not None else None}) + '\n')
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
import shutil
manifest = {
    'base_model': BASE, 'adapter': str(ADAPTER), 'train_rows': len(train_rows),
    'eval_pairs': len(eval_rows) // 2, 'ethics_split': 'test', 'ethics_n': 100,
    'train_sha256': hashlib.sha256(TRAIN.read_bytes()).hexdigest(),
    'paired_eval_sha256': hashlib.sha256(EVAL.read_bytes()).hexdigest(),
    'adapter_sha256': hashlib.sha256((ADAPTER / 'adapter_model.safetensors').read_bytes()).hexdigest(),
    'epochs': 3, 'seed': 42, 'checkpoint_selection': 'final epoch; no evaluation-set selection',
    'torch': torch.__version__,
}
(OUT / 'experiment.json').write_text(json.dumps(manifest, indent=2))
shutil.copy2(ADAPTER / 'training_log.json', OUT / 'training_log.json')
(OUT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
results = shutil.make_archive('/content/clause_order_baselines', 'zip', root_dir=OUT)
weights = '/content/qwen05b-cot-sft-clause-order.zip'
with zipfile.ZipFile(weights, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in ADAPTER.iterdir():
        if path.is_file():
            z.write(path, path.name)
files.download(results)
files.download(weights)
print('Download both ZIPs. Critic scoring runs locally after extraction.')


## Local critic (after downloading)
Extract `clause_order_baselines.zip` into `data/clause_order_baselines` in your local repository, then run:
```bash
python3 evaluation/score_clause_order.py --dir data/clause_order_baselines
```
Uses your local Azure configuration. The critic sees only reasoning, never the model identity, predicted answer, or constructed target. Its cache makes reruns resumable and reuses identical fixed-CoT judgments across base and LoRA.

Inspect `clause_order_baselines_summary.json`: compare both order accuracies, paired label flips, both-members-correct, free-generation rule-following, parsing rate, and ETHICS accuracy. Mixed/unclear and unparsed outputs count as non-follow in the all-example rate. These baselines do not establish transfer of the frozen S1+voice direction; that is the next experiment.
